# Test 1 (main text): the 15 weights drawn at random, three levels, two bases

All computing is in `p1_tests.py`, which imports `ws_core.py` (the code of the supplement). **Run All** reads the data again: changed data give new folders under `results_p1/`, unchanged data resume and only display. Results are reported, not read as pass or fail.

Weights: the 15 weights of Table A.11 as one flat list (values as the thesis code holds them: 1/45 and 1/3 where the table prints 0.022 and 0.333). Draws: `results_p1/weight_draws_1000.csv`, 1 000 rows of 15 factors exp(N(0, 0.35²)), seed 42 (log in `weight_draws_1000.json`). Band: 10 000 shuffles per run, seed 42, the same shuffles in every run, edge = 2.5th percentile; margin = band edge − ΔQ; holds = margin > 0.

Cohort 2020-23 against the two earliest cohorts. Bases: all designs; actively protected (ΔQ against the all-designs baseline, band rebuilt on all designs with the 2020-23 block cut at the actively protected size). (a) the 1 000 draws, band rebuilt in each; (b) one run with all 15 weights equal.

In [ ]:
%matplotlib inline
import pandas as pd, ws_core as ws, p1_tests as p1
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.float_format', lambda x: f'{x:.4f}')
DATASETS = ('v1', 'thesis')    # eVTOL_dataset_v1 as it is now, then the thesis as delivered
P = {d: p1.prepare(d) for d in DATASETS}   # reads each dataset with the thesis code; unchanged data are not rewritten

## 1. Baseline check
The thesis data must give ΔQ = −0.028 and band edge = −0.020 on all designs (level A, 2020-23); the run stops if not. Then the thesis code against this code at the three levels of each dataset.

In [ ]:
ws.baseline_check(ws.build_inputs('thesis', verbose=False))

In [ ]:
for d, p in P.items():
    print(d); display(p1.baseline_checks(p).query("inputs in ['L1', 'L2', 'L3']"))

## 2. How the level enters Q
Q is the mean Gower distance over the designs of a cohort and does not use the level. The level only decides which designs enter: those the level can name (`frames.archetype_key`). L2 (A·W·T) and L3 (A·U) leave out one design of 2020-23 (GB202302720D0_ua1, no propulsor count and no tilt); L2 also one of 2024-26, which changes the pool the shuffles are drawn from. So L2 and L3 differ from L1 by those designs only.

In [ ]:
for d, p in P.items():
    print(d); display(p1.designs_per_cohort(p).drop(index='F'))

## 3. Hardware and timing
20 runs on all CPU cores and on the GPU; the faster is used. Estimate before launching.

In [ ]:
T = {d: p1.timing(p) for d, p in P.items()}
T['v1']

## 4. Runs
Baseline and equal-weights runs, then the 1 000 draws per level; saved every 50 draws, resumable. Progress shows the running share that holds.

In [ ]:
for d, p in P.items():
    print(f'================ {d}')
    p1.run_test1(p, backend=T[d].loc[0, 'used'])

## 5. Results
`test1_summary.csv`: per level and base, (a) the share of draws in which the disparity verdict holds with its Wilson 95 % interval, the median margin and its 5th and 95th percentile; (b) the equal-weights run; and the Table A.11 run beside them. `test1_draws.csv`: one row per draw, level and base.

In [ ]:
S1 = {d: p1.summary_test1(p) for d, p in P.items()}
S1['v1']

In [ ]:
fig = p1.figure_test1(P['v1'])

In [ ]:
S1['thesis']

In [ ]:
fig = p1.figure_test1(P['thesis'])

In [ ]:
for d, p in P.items():
    print(p['out'])